# Raw Data Exploration
3 JSON dosyasını incele, duplicate'leri temizle, tek bir temiz JSON üret.

In [1]:
import json
import pandas as pd
from pathlib import Path

DATA_DIR = Path('../backend/data/raw')
OUT_DIR  = Path('../backend/data')
OUT_DIR.mkdir(exist_ok=True)

## 1. Dosyaları Yükle

In [2]:
files = list(DATA_DIR.glob('*.json'))
frames = {}
for f in files:
    data = json.loads(f.read_text(encoding='utf-8'))
    df = pd.DataFrame(data)
    df['_source_file'] = f.stem
    df['date'] = pd.to_datetime(df['date'])
    frames[f.stem] = df
    print(f'{f.name}: {len(df)} satır')

raw = pd.concat(frames.values(), ignore_index=True)
raw = raw[raw['productSlug'] == 'apollo-io'].copy()
print(f'\nToplam Apollo.io (duplicate dahil): {len(raw)} satır')

dataset_g2-reviews-scraper-highest_2026-06-29_15-01-28-325.json: 770 satır
dataset_g2-reviews-scraper-lowest_2026-06-29_14-35-25-097.json: 770 satır
dataset_g2-reviews-scraper-most-recent_2026-06-29_14-26-15-073.json: 759 satır

Toplam Apollo.io (duplicate dahil): 2299 satır


## 2. Dosya Bazlı Karşılaştırma
Scraper filtreleri gerçekten farklı review'lar mı döndürdü?

In [3]:
# Her dosyadaki unique reviewId setleri
id_sets = {}
for name, df in frames.items():
    apollo = df[df['productSlug'] == 'apollo-io']
    id_sets[name] = set(apollo['reviewId'].unique())
    short = name.split('scraper-')[-1][:25]
    print(f'{short}: {len(id_sets[name])} unique reviewId')

highest_2026-06-29_15-01-: 770 unique reviewId
lowest_2026-06-29_14-35-2: 770 unique reviewId
most-recent_2026-06-29_14: 759 unique reviewId


In [4]:
# Dosyalar arası overlap
keys = list(id_sets.keys())
for i in range(len(keys)):
    for j in range(i+1, len(keys)):
        a, b = keys[i], keys[j]
        overlap = id_sets[a] & id_sets[b]
        only_a  = id_sets[a] - id_sets[b]
        only_b  = id_sets[b] - id_sets[a]
        sa = a.split('scraper-')[-1][:20]
        sb = b.split('scraper-')[-1][:20]
        print(f'\n{sa} vs {sb}')
        print(f'  Ortak           : {len(overlap)}')
        print(f'  Sadece 1. dosya : {len(only_a)}')
        print(f'  Sadece 2. dosya : {len(only_b)}')


highest_2026-06-29_1 vs lowest_2026-06-29_14
  Ortak           : 770
  Sadece 1. dosya : 0
  Sadece 2. dosya : 0

highest_2026-06-29_1 vs most-recent_2026-06-
  Ortak           : 759
  Sadece 1. dosya : 11
  Sadece 2. dosya : 0

lowest_2026-06-29_14 vs most-recent_2026-06-
  Ortak           : 759
  Sadece 1. dosya : 11
  Sadece 2. dosya : 0


In [5]:
# Her dosyada starRating ve tarih aralığı — filtreler çalıştı mı?
for name, df in frames.items():
    apollo = df[df['productSlug'] == 'apollo-io'].copy()
    short = name.split('scraper-')[-1][:25]
    print(f'\n--- {short} ---')
    print('starRating:', apollo['starRating'].value_counts().sort_index().to_dict())
    print(f'Tarih aralığı: {apollo["date"].min().date()} → {apollo["date"].max().date()}')


--- highest_2026-06-29_15-01- ---
starRating: {0.0: 13, 0.5: 4, 1.0: 2, 1.5: 2, 2.0: 2, 2.5: 9, 3.0: 11, 3.5: 53, 4.0: 163, 4.5: 206, 5.0: 305}
Tarih aralığı: 2025-07-24 → 2026-06-28

--- lowest_2026-06-29_14-35-2 ---
starRating: {0.0: 13, 0.5: 4, 1.0: 2, 1.5: 2, 2.0: 2, 2.5: 9, 3.0: 11, 3.5: 53, 4.0: 163, 4.5: 206, 5.0: 305}
Tarih aralığı: 2025-07-24 → 2026-06-28

--- most-recent_2026-06-29_14 ---
starRating: {0.0: 13, 0.5: 4, 1.0: 2, 1.5: 2, 2.0: 2, 2.5: 9, 3.0: 11, 3.5: 53, 4.0: 161, 4.5: 204, 5.0: 298}
Tarih aralığı: 2025-07-29 → 2026-06-28


In [6]:
# reviewId sayısal aralığı — hep aynı noktadan mı başlıyor?
for name, df in frames.items():
    apollo = df[df['productSlug'] == 'apollo-io']
    ids = apollo['reviewId'].astype(int).sort_values()
    short = name.split('scraper-')[-1][:25]
    print(f'{short}: min={ids.min()}  max={ids.max()}  medyan={ids.median():.0f}')

highest_2026-06-29_15-01-: min=5300916  max=13041581  medyan=12082331
lowest_2026-06-29_14-35-2: min=5300916  max=13041581  medyan=12082331
most-recent_2026-06-29_14: min=5300916  max=13041581  medyan=12093155


## 3. Duplicate Analizi

In [7]:
total      = len(raw)
unique_ids = raw['reviewId'].nunique()
dupes      = total - unique_ids

print(f'Toplam satır    : {total}')
print(f'Unique reviewId : {unique_ids}')
print(f'Duplicate sayısı: {dupes}  (%{dupes/total*100:.1f})')
print()

# Bir ID kaç farklı dosyada görünüyor?
id_file_count = raw.groupby('reviewId')['_source_file'].nunique()
print('Bir review kaç dosyada görünüyor:')
print(id_file_count.value_counts().sort_index())

Toplam satır    : 2299
Unique reviewId : 770
Duplicate sayısı: 1529  (%66.5)

Bir review kaç dosyada görünüyor:
_source_file
2     11
3    759
Name: count, dtype: int64


## 4. Alan Kalitesi

In [8]:
uniq = raw.drop_duplicates('reviewId')

print('starRating dağılımı (unique):')
print(uniq['starRating'].value_counts().sort_index())
print()
print(f'Tarih aralığı: {uniq["date"].min().date()} → {uniq["date"].max().date()}')
print()
print('Null değer sayıları:')
print(uniq.isnull().sum()[uniq.isnull().sum() > 0])

starRating dağılımı (unique):
starRating
0.0     13
0.5      4
1.0      2
1.5      2
2.0      2
2.5      9
3.0     11
3.5     53
4.0    163
4.5    206
5.0    305
Name: count, dtype: int64

Tarih aralığı: 2025-07-24 → 2026-06-28

Null değer sayıları:
text                   2
reviewerInfo         232
reviewerTitle         38
validatedMethod      770
reviewerAvatarUrl    234
resolvedUrl          730
dtype: int64


In [9]:
def parse_company_size(info):
    if not info:
        return None
    for item in info:
        if any(k in item.lower() for k in ['emp', 'business', 'enterprise', 'mid-market']):
            return item
    return info[0] if info else None

raw['company_size'] = raw['reviewerInfo'].apply(parse_company_size)
print('company_size dağılımı (unique):')
print(raw.drop_duplicates('reviewId')['company_size'].value_counts())

company_size dağılımı (unique):
company_size
Small-Business (50 or fewer emp.)    301
Mid-Market (51-1000 emp.)            218
Enterprise (> 1000 emp.)              19
Name: count, dtype: int64


## 5. Temizle & Kaydet

In [10]:
def split_text(text):
    if not text or '|' not in str(text):
        return pd.Series({'likes': text, 'dislikes': None})
    parts = str(text).split('|', 1)
    return pd.Series({'likes': parts[0].strip(), 'dislikes': parts[1].strip()})

clean = raw.drop_duplicates(subset='reviewId', keep='first').copy()
clean[['likes', 'dislikes']] = clean['text'].apply(split_text)

drop_cols = ['text', 'markdownContent', 'reviewerAvatarUrl', 'reviewerMonogram',
             '_source_file', 'type', 'resolvedUrl', 'validatedMethod']
clean = clean.drop(columns=[c for c in drop_cols if c in clean.columns])
clean['date'] = clean['date'].dt.strftime('%Y-%m-%d')

print(f'Temiz kayıt sayısı: {len(clean)}')
print(f'Kolonlar: {clean.columns.tolist()}')
clean.head(2)

Temiz kayıt sayısı: 770
Kolonlar: ['date', 'title', 'reviewId', 'starRating', 'productName', 'productSlug', 'incentivized', 'reviewSource', 'reviewerInfo', 'reviewerName', 'reviewerTitle', 'validatedReviewer', 'verifiedCurrentUser', 'company_size', 'likes', 'dislikes']


,date,title,reviewId,starRating,productName,productSlug,incentivized,reviewSource,reviewerInfo,reviewerName,reviewerTitle,validatedReviewer,verifiedCurrentUser,company_size,likes,dislikes
0,2026-06-28,Really Good for Mapping and Extracting Contacts,13041581,5.0,apollo-io,apollo-io,True,Organic,[Small-Business (50 or fewer emp.)],Kunal P.,Analyst,True,True,Small-Business (50 or fewer emp.),Really good for mapping and taking out contacts,"It’s been kind of slow over the last few days,..."
1,2026-06-25,"Great for Leads and Company Insights, but Emai...",13029014,3.5,apollo-io,apollo-io,False,Organic,"[Information Technology and Services, Mid-Mark...",Eldho M.,Associate Manager - Business Development,True,False,Mid-Market (51-1000 emp.),"Good for leads, emails and company insights",some times the email addresses are not working.


In [11]:
out_path = OUT_DIR / 'reviews_clean.json'
clean.to_json(out_path, orient='records', force_ascii=False, indent=2)
print(f'Kaydedildi → {out_path}')
print(f'Toplam unique review: {len(clean)}')

Kaydedildi → ..\backend\data\reviews_clean.json
Toplam unique review: 770


## 6. Clean Data İncelemesi

In [12]:
df = pd.read_json(OUT_DIR / 'reviews_clean.json')
df['date'] = pd.to_datetime(df['date'])
print(f'Kayıt sayısı : {len(df)}')
print(f'Kolon sayısı : {len(df.columns)}')
print(f'Kolonlar     : {df.columns.tolist()}')

Kayıt sayısı : 770
Kolon sayısı : 16
Kolonlar     : ['date', 'title', 'reviewId', 'starRating', 'productName', 'productSlug', 'incentivized', 'reviewSource', 'reviewerInfo', 'reviewerName', 'reviewerTitle', 'validatedReviewer', 'verifiedCurrentUser', 'company_size', 'likes', 'dislikes']


In [13]:
# Null / boş alan raporu
null_counts = df.isnull().sum()
null_pct    = (null_counts / len(df) * 100).round(1)
report = pd.DataFrame({'null_adet': null_counts, 'null_%': null_pct})
print(report[report['null_adet'] > 0])

               null_adet  null_%
reviewerInfo         232    30.1
reviewerTitle         38     4.9
company_size         232    30.1
likes                  2     0.3
dislikes               2     0.3


In [14]:
# starRating dağılımı — 0.0 ve 0.5 şüpheli
print('starRating dağılımı:')
print(df['starRating'].value_counts().sort_index())
print()
supheli = df[df['starRating'] < 1.0]
print(f'0.0 / 0.5 yıldız olan review sayısı: {len(supheli)}')
print()
print('Örnek şüpheli review:')
print(supheli[['reviewId', 'starRating', 'title', 'likes', 'dislikes']].head(3).to_string())

starRating dağılımı:
starRating
0.0     13
0.5      4
1.0      2
1.5      2
2.0      2
2.5      9
3.0     11
3.5     53
4.0    163
4.5    206
5.0    305
Name: count, dtype: int64

0.0 / 0.5 yıldız olan review sayısı: 17

Örnek şüpheli review:
    reviewId  starRating                                           title                                                                                                                                                                                                                                                                                likes                                                                                                                                                                                                                                                     dislikes
29  12881616         0.0  Begging you not to buy this, save your GTM org  There is nothing I like about this platform. I am an AE expected to make 20+ dia

In [15]:
# Aylık review dağılımı
df['month'] = df['date'].dt.to_period('M')
print('Aylık review sayısı:')
print(df['month'].value_counts().sort_index())

Aylık review sayısı:
month
2025-07     32
2025-08    104
2025-09     94
2025-10     68
2025-11     41
2025-12     79
2026-01     64
2026-02     36
2026-03    104
2026-04     73
2026-05     52
2026-06     23
Freq: M, Name: count, dtype: int64


In [16]:
# likes / dislikes metin kalitesi
df['likes_len']    = df['likes'].fillna('').str.len()
df['dislikes_len'] = df['dislikes'].fillna('').str.len()

print(f'likes boş (None)     : {df["likes"].isna().sum()}')
print(f'likes çok kısa (<10) : {(df["likes_len"].between(1,9)).sum()}')
print(f'dislikes boş (None)  : {df["dislikes"].isna().sum()}')
print(f'dislikes çok kısa    : {(df["dislikes_len"].between(1,9)).sum()}')
print()
print('likes uzunluk istatistikleri:')
print(df['likes_len'].describe().round(1))
print()
print('dislikes uzunluk istatistikleri:')
print(df['dislikes_len'].describe().round(1))

likes boş (None)     : 2
likes çok kısa (<10) : 1
dislikes boş (None)  : 2
dislikes çok kısa    : 5

likes uzunluk istatistikleri:
count     770.0
mean      244.0
std       196.5
min         0.0
25%        85.5
50%       191.5
75%       353.0
max      1240.0
Name: likes_len, dtype: float64

dislikes uzunluk istatistikleri:
count    770.0
mean     168.0
std      139.4
min        0.0
25%       67.0
50%      120.0
75%      236.0
max      946.0
Name: dislikes_len, dtype: float64


In [17]:
# Şirket büyüklüğü ve sektör ayrıştır
print('company_size dağılımı:')
print(df['company_size'].value_counts(dropna=False))
print()

def parse_industry(info):
    if not info:
        return None
    for item in info:
        if not any(k in item.lower() for k in ['emp.', 'small-business', 'enterprise', 'mid-market']):
            return item
    return None

df['industry'] = df['reviewerInfo'].apply(parse_industry)
print('industry dağılımı (top 15):')
print(df['industry'].value_counts(dropna=False).head(15))

company_size dağılımı:
company_size
Small-Business (50 or fewer emp.)    301
NaN                                  232
Mid-Market (51-1000 emp.)            218
Enterprise (> 1000 emp.)              19
Name: count, dtype: int64

industry dağılımı (top 15):
industry
NaN                                    666
Information Technology and Services     38
Computer Software                       13
Staffing and Recruiting                 11
Marketing and Advertising                8
Financial Services                       4
Insurance                                3
Consulting                               2
Internet                                 2
Market Research                          2
Graphic Design                           2
Mining & Metals                          1
Health, Wellness and Fitness             1
Cosmetics                                1
Broadcast Media                          1
Name: count, dtype: int64


In [18]:
# Her rating grubundan örnek review
for rating, label in [(5.0, '5 yıldız'), (3.5, '3.5 yıldız'), (1.0, '1 yıldız ve altı')]:
    subset = df[df['starRating'] <= rating] if label == '1 yıldız ve altı' else df[df['starRating'] == rating]
    if len(subset) == 0:
        continue
    row = subset.sample(1).iloc[0]
    print(f'=== {label} ===')
    print(f'Başlık  : {row["title"]}')
    print(f'likes   : {str(row["likes"])[:150]}')
    print(f'dislikes: {str(row["dislikes"])[:150]}')
    print()

=== 5 yıldız ===
Başlık  : Seamless Integration and Stellar Email Deliverability
likes   : I like that Apollo.io is relatively easy to use with a straightforward setup that wasn't hard or extensive when we onboarded it. Even though we had so
dislikes: I do think that the onboarding could have been given a little bit more love. Having to schedule a one-hour call with someone who didn't seem super pre

=== 3.5 yıldız ===
Başlık  : Huge Database but Sluggish Performance
likes   : I find Apollo.io's people and company search extremely easy and useful with its various keywords and filters. The extensive database is a significant 
dislikes: I find the initial setup of Apollo.io a bit confusing, although it becomes manageable with time. The speed is also a concern as it slows my system dow

=== 1 yıldız ve altı ===
Başlık  : Pricey Changes Diminish Value for Small Businesses
likes   : I initially liked Apollo.io because it brought multiple outreach tools together at a great price for start-up